# JASSIM AI — Google Colab GPU Serverدفتر تشغيل تجريبي لخادم JASSIM AI. يحفظ SQLite في Google Drive، ويشغّل Ollama والنموذج، ثم يفتح نفق HTTPS مؤقتاً عبر Cloudflare.**ملاحظات:** جلسات Colab المجانية مؤقتة وGPU غير مضمون. توجد خلية Watchdog لإعادة تشغيل العمليات المتعطلة، لكنها لا تمنع إيقاف جلسة Colab. لا تشارك رابط النفق أو رمز API مع الآخرين. لا تضع أسراراً في GitHub.من قائمة Runtime اختر Change runtime type ثم GPU إن كان متاحاً.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, subprocess, time, json, secrets, requests, re, threading

PROJECT = Path('/content/JASSIM-AI-Android')
DATA_DIR = Path('/content/drive/MyDrive/JASSIM_AI/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Drive mounted. Persistent database directory:', DATA_DIR)

## 1. تنزيل ملفات خادم JASSIMنستخدم نسخة المستودع الحالية دون تعديل ملفات التطبيق أو الذاكرة الموجودة فيه.

In [ ]:
if not PROJECT.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/stevennpllk-crypto/-JASSIM-AI-Android.git',str(PROJECT)], check=True)
else:
    subprocess.run(['git','-C',str(PROJECT),'pull','--ff-only'], check=True)
SERVER = PROJECT / 'backend' / 'server.py'
assert SERVER.exists(), 'backend/server.py is missing'
print('Server source ready:', SERVER)

## 2. إعداد رمز الحمايةأنشئ رمزاً عشوائياً أو أدخل رمزاً تملكه. سيُستخدم في التطبيق والموقع. لا ترسله في المحادثة ولا تحفظه في GitHub.

In [ ]:
API_TOKEN = secrets.token_urlsafe(48)
print('PRIVATE API TOKEN — copy privately into the Android app:')
print(API_TOKEN)

os.environ['JASSIM_API_TOKEN'] = API_TOKEN
os.environ['JASSIM_CORS_ORIGIN'] = 'https://stevennpllk-crypto.github.io'
os.environ['JASSIM_HOST'] = '127.0.0.1'
os.environ['JASSIM_PORT'] = '8787'
os.environ['JASSIM_DATA_DIR'] = str(DATA_DIR)
os.environ['JASSIM_OLLAMA_URL'] = 'http://127.0.0.1:11434'
os.environ['JASSIM_OLLAMA_MODEL'] = 'qwen3.5:4b'
os.environ['JASSIM_OLLAMA_KEEP_ALIVE'] = '-1'
os.environ['JASSIM_NUM_CTX'] = '8192'
os.environ['JASSIM_NUM_PREDICT'] = '2048'
os.environ['JASSIM_HISTORY_MESSAGES'] = '20'
os.environ['JASSIM_HISTORY_CHAR_BUDGET'] = '24000'
os.environ['JASSIM_REQUEST_TIMEOUT'] = '300'

## 3. تثبيت وتشغيل Ollama
نستخدم Qwen 3.5 4B كخط أساس متوازن. يمكن اختيار نموذج أكبر بتغيير المتغير إذا كانت ذاكرة GPU/RAM تسمح. يبقى النموذج محمّلاً طوال عمل Ollama (`keep_alive=-1`)، لكن Colab قد ينهي الجلسة تلقائياً.

In [ ]:
install = subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, text=True, capture_output=True)
print(install.stdout[-2000:])
if install.returncode != 0:
    print(install.stderr[-2000:])
    raise RuntimeError('Ollama installation failed')

ollama_log = open('/content/ollama.log','a')
ollama_proc = subprocess.Popen(['ollama','serve'], stdout=ollama_log, stderr=subprocess.STDOUT)
for _ in range(45):
    try:
        requests.get('http://127.0.0.1:11434/api/tags', timeout=3).raise_for_status()
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Ollama did not start. Inspect /content/ollama.log')

MODEL = os.environ['JASSIM_OLLAMA_MODEL']
pull = subprocess.run(['ollama','pull',MODEL], text=True)
if pull.returncode != 0:
    raise RuntimeError('Model download failed')
print('Model downloaded:', MODEL)

## 4. تشغيل JASSIM API مع ذاكرة Driveتُحفظ قاعدة البيانات داخل مجلد JASSIM_AI في Drive. لا تحذف هذا المجلد إذا أردت الاحتفاظ بالمحادثات.

In [ ]:
api_log = open('/content/jassim-api.log','a')
api_proc = subprocess.Popen(['python',str(SERVER)], cwd=str(PROJECT/'backend'), env=os.environ.copy(), stdout=api_log, stderr=subprocess.STDOUT)
for _ in range(45):
    try:
        ready = requests.get('http://127.0.0.1:8787/ready', timeout=3)
        if ready.status_code == 200:
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError('JASSIM API is not ready. Inspect /content/jassim-api.log')
print(json.dumps(ready.json(), ensure_ascii=False, indent=2))

## 5. اختبار المحرك والحمايةهذا الاختبار يرسل رسالة تجريبية إلى النموذج ويحفظها في سجل المحادثة.

In [ ]:
headers = {'Authorization':'Bearer '+API_TOKEN,'Content-Type':'application/json'}
test = requests.post('http://127.0.0.1:8787/api/chat', headers=headers, json={'message':'أجب بالعربية: اشرح باختصار لماذا تتغير الفصول الأربعة.','conversation_id':'colab-check'}, timeout=300)
print('HTTP:', test.status_code)
print(json.dumps(test.json(), ensure_ascii=False, indent=2))
test.raise_for_status()
assert test.json().get('engine') == 'ollama', 'The model did not generate this answer'

## 6. فتح رابط HTTPS مؤقتسنشغّل Cloudflare Quick Tunnel. الرابط يتغير عند إعادة تشغيل الجلسة، وهو ليس استضافة دائمة.

In [ ]:
deb = '/content/cloudflared.deb'
subprocess.run(['bash','-lc',f'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O {deb} && dpkg -i {deb}'], check=True)
cf_log = open('/content/cloudflared.log','a')
cf_proc = subprocess.Popen(['cloudflared','tunnel','--no-autoupdate','--url','http://127.0.0.1:8787'], stdout=cf_log, stderr=subprocess.STDOUT, text=True)
def find_tunnel_url():
    log_text = Path('/content/cloudflared.log').read_text(errors='ignore')
    matches = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
    return matches[-1] if matches else None

tunnel_url = None
for _ in range(60):
    time.sleep(2)
    tunnel_url = find_tunnel_url()
    if tunnel_url:
        break
if not tunnel_url:
    raise RuntimeError('Tunnel URL not found. Inspect /content/cloudflared.log')
print('HTTPS API URL:', tunnel_url)
print('Health:', requests.get(tunnel_url+'/health', timeout=20).json())

## 7. ربط التطبيق والموقعفي إعدادات تطبيق Android ولوحة التحكم:- Server URL: انسخ رابط HTTPS الذي ظهر في الخلية السابقة.- API Token: انسخ الرمز الخاص الذي وُلّد في خلية الحماية.احتفظ بالرابط والرمز بشكل خاص. لا تنشرهما ولا ترسلهما إلى أشخاص آخرين. عند انتهاء جلسة Colab أعد تشغيل الخلايا؛ قد تحصل على رابط جديد، لكن بيانات SQLite تبقى في Drive.**مهم:** لا تترك الخلية الأخيرة دون مراقبة؛ إيقاف جلسة Colab يوقف النموذج والخادم والنفق.

In [ ]:
print('API URL:', tunnel_url)
print('API Token was printed in the security cell above; keep it private.')
print('Persistent database folder:', DATA_DIR)
print('Keep this Colab runtime active while using JASSIM.')

## 8. مراقبة وإعادة تشغيل تلقائية (Watchdog)شغّل الخلية التالية مرة واحدة. تحاول إعادة تشغيل Ollama وAPI والنفق إذا تعطلت العمليات أو تكرر فشل الفحص. عند تغير رابط النفق اطبعه مجدداً وحدّث إعدادات التطبيق. هذه الحماية تعمل فقط ما دامت جلسة Colab نفسها حية؛ لا تمنع فصل Colab.

In [ ]:
        matches = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
